In [ ]:
#visualisation Grad-CAM
# On cible la dernière couche de convolution du ResNet-18
target_layer = model.layer4[-1] 

# On crée un dictionnaire pour stocker les activations et les gradients
gradients = {}
activations = {}

def get_gradients(name):
    def hook(model, input, output):
        gradients[name] = output[0]
    return hook

def get_activations(name):
    def hook(model, input, output):
        activations[name] = output
    return hook

# On attache les "hooks" (les crochets) au modèle
target_layer.register_forward_hook(get_activations('value'))
target_layer.register_backward_hook(get_gradients('value'))

import cv2
import torch.nn.functional as F

def generate_gradcam(img_tensor, label_idx):
    model.eval()
    # 1. Forward pass
    output = model(img_tensor.unsqueeze(0).to(device))
    
    # 2. Backward pass sur la classe prédite
    model.zero_grad()
    output[:, label_idx].backward()
    
    # 3. Calcul du Grad-CAM
    grads = gradients['value']
    pooled_grads = torch.mean(grads, dim=[0, 2, 3])
    
    act = activations['value'][0]
    for i in range(act.shape[0]):
        act[i, :, :] *= pooled_grads[i]
        
    heatmap = torch.mean(act, dim=0).cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0) # ReLU sur la heatmap
    heatmap /= np.max(heatmap) # Normalisation
    
    return heatmap

In [ ]:
# 1. On récupère un batch du loader VinDr
images, labels = next(iter(test_loader_vindr))

# 2. On choisit une image précise (ex: la première du batch)
index = 31
image_test = images[index]  # C'est le tenseur [3, 900, 900] normalisé
label_test = labels[index]  # C'est le vrai label (0 ou 1)

# 3. On prépare l'image "originale" pour l'affichage final
# Comme image_test est normalisée (moyenne/std), elle est moche à l'œil.
# On la "dé-normalise" pour que l'affichage soit clair.
image_original = image_test.permute(1, 2, 0).cpu().numpy()
image_original = (image_original - image_original.min()) / (image_original.max() - image_original.min())

In [ ]:
for i in range(32):
    print(labels[i])

In [ ]:
heatmap = generate_gradcam(image_test, label_test)
print(f"label : {label_test}")
heatmap_resized = cv2.resize(heatmap, (672, 672))
heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)

# Superposition
superimposed_img = heatmap_color * 0.003 + image_original * 0.997
#plt.imshow(heatmap_color)
#plt.imshow(image_original)
plt.imshow(superimposed_img)

In [ ]:
plt.imshow(heatmap_color)


In [ ]:
plt.imshow(image_original)


In [ ]:
model.eval()
trouve = False

with torch.no_grad():
    for images, labels in test_loader_vindr: # On parcourt tous les batchs
        print("1")
        outputs = model(images.to(device))
        _, preds = torch.max(outputs, 1)
        
        # On cherche les indices d'erreurs dans CE batch
        errors = (preds.cpu() != labels)
        error_indices = torch.where(errors)[0]
        
        if len(error_indices) > 0:
            idx = error_indices[0] # On prend la première erreur du batch
            image_erreur = images[idx]
            label_vrai = labels[idx]
            label_predit = preds[idx]
            print(idx)
            print(f"✅ Erreur trouvée ! Vrai: {label_vrai}, Prédit: {label_predit}")
            trouve = True
            break # On arrête de chercher, on a notre image
            
    if not trouve:
        print("❌ Aucune erreur trouvée dans tout le dataset (peu probable !)")

In [ ]:
import numpy as np
from PIL import Image

def dynamic_crop(img):
    # 1. Convertir en array pour trouver le sein
    np_img = np.array(img)
    
    # 2. Trouver les pixels non-noirs (seuil à 5 ou 10 pour le bruit)
    mask = np_img > 10
    coords = np.argwhere(mask)
    
    if coords.size == 0:
        return img # Image vide, on renvoie l'original
    
    # 3. Récupérer les coins du rectangle
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0)
    
    # 4. Découper et repasser en format PIL pour PyTorch
    return Image.fromarray(np_img[y0:y1, x0:x1])

In [ ]:
# 1. On récupère une image brute du dataset (pas du loader)
# Supposons que ton dataset s'appelle 'dataset_test'
img_brute, label = data_set[0] # img_brute est ici au format PIL (image d'origine)

# 2. On applique le crop
img_rognée = dynamic_crop(img_brute)

# 3. On applique manuellement tes transforms pour pouvoir la donner au modèle
# (Remplace par tes propres transforms : Resize, ToTensor, Normalize)
img_prete = data_set(img_rognée) 

# 4. Maintenant tu peux générer ton Grad-CAM sur 'img_prete'
heatmap = generate_gradcam(img_prete, label_idx=1)